# Notebook-first application walkthrough

**Problem / objective:** Classify product/wine quality groups with an interpretable distance-based model and show why feature scaling matters for KNN.

**Decision / solution:** Route confident cases automatically and send ambiguous nearest-neighbour cases to review with the neighbour evidence visible.

This front section is intentionally analysis-first. It uses direct notebook code for inspection, EDA, visualisation and evidence review. The original notebook work is preserved below, followed by modular production code where that adds engineering evidence.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
PROJECT_SLUG = 'knn_product_quality'
ROOT = Path.cwd()
if not (ROOT / 'projects').exists():
    candidate = ROOT.parent.parent if ROOT.name == PROJECT_SLUG else ROOT
    if (candidate / 'projects').exists():
        ROOT = candidate
PROJECT = ROOT / 'projects' / PROJECT_SLUG
if not PROJECT.exists() and Path.cwd().name == PROJECT_SLUG:
    PROJECT = Path.cwd()
    ROOT = PROJECT.parent.parent
assert PROJECT.exists(), f'Project directory not found: {PROJECT}'
print('Repository root:', ROOT.resolve())
print('Project:', PROJECT.resolve())


## 1. Find the real data and retained evidence

Instead of hiding the dataset behind a helper function, start by seeing what the project actually ships: raw/small data, fixtures, outputs, results and verified evidence. External large datasets remain reproducibly downloadable from the documented source.


In [ ]:
candidate_files = []
for pattern in ('*.csv', '*.parquet', '*.json', '*.tsv', '*.txt'):
    candidate_files.extend(PROJECT.rglob(pattern))
verified_dir = ROOT / 'verified' / PROJECT_SLUG
if verified_dir.exists():
    for pattern in ('*.csv', '*.parquet', '*.json', '*.tsv', '*.txt'):
        candidate_files.extend(verified_dir.rglob(pattern))
candidate_files = sorted({p.resolve() for p in candidate_files if p.is_file()})
file_inventory = pd.DataFrame({
    'file': [str(p.relative_to(ROOT)) if ROOT in p.parents else str(p) for p in candidate_files],
    'suffix': [p.suffix.lower() for p in candidate_files],
    'size_kb': [round(p.stat().st_size / 1024, 1) for p in candidate_files],
})
display(file_inventory.head(40))
print(f'Inspectable local data/evidence files: {len(file_inventory):,}')


## 2. Direct tabular data audit

The code below deliberately avoids a project-specific wrapper. It opens the first sensible local tabular asset, shows its schema and quality profile, and makes the data issues visible before modelling. If the full raw dataset is external, run the project's documented download cell/entry point first and rerun this section.


In [ ]:
tabular_candidates = [p for p in candidate_files if p.suffix.lower() in {'.csv', '.tsv', '.parquet'}]
preferred = [p for p in tabular_candidates if not any(token in p.name.lower() for token in ('metric', 'summary', 'verification'))]
tabular_path = (preferred or tabular_candidates or [None])[0]
df = None
if tabular_path is not None:
    if tabular_path.suffix.lower() == '.parquet':
        df = pd.read_parquet(tabular_path)
    else:
        sep = '\t' if tabular_path.suffix.lower() == '.tsv' else ','
        df = pd.read_csv(tabular_path, sep=sep, nrows=200_000)
    print('Loaded:', tabular_path)
    print('Shape:', df.shape)
    display(df.head())
    audit = pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'missing': df.isna().sum(),
        'missing_pct': (100 * df.isna().mean()).round(2),
        'unique': df.nunique(dropna=False),
    }).sort_values(['missing_pct', 'unique'], ascending=[False, False])
    display(audit.head(30))
    print('Duplicate rows:', int(df.duplicated().sum()))
else:
    print('No local CSV/TSV/Parquet found yet. Use the project README/run path to download or build the documented dataset, then rerun this audit.')


## 3. Exploratory data analysis and visualisation

These plots are intentionally created in the notebook rather than described in prose. They expose distribution, missingness, scale, category balance and numeric relationships before any final model decision.


In [ ]:
if df is not None and len(df):
    missing_pct = (100 * df.isna().mean()).sort_values(ascending=False).head(20)
    missing_pct = missing_pct[missing_pct > 0]
    if len(missing_pct):
        plt.figure(figsize=(10, 4))
        missing_pct.plot(kind='bar')
        plt.title('Missing values by feature (%)')
        plt.ylabel('Missing %')
        plt.xticks(rotation=60, ha='right')
        plt.tight_layout()
        plt.show()

    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()[:8]
    for col in numeric_cols:
        series = pd.to_numeric(df[col], errors='coerce').dropna()
        if len(series):
            plt.figure(figsize=(8, 4))
            plt.hist(series, bins=30, alpha=0.8)
            plt.axvline(series.median(), linestyle='--', label=f'median={series.median():.2f}')
            plt.title(f'Distribution: {col}')
            plt.xlabel(col)
            plt.ylabel('Count')
            plt.legend()
            plt.tight_layout()
            plt.show()

    categorical_cols = [c for c in df.columns if c not in numeric_cols and df[c].nunique(dropna=False) <= 30][:4]
    for col in categorical_cols:
        counts = df[col].fillna('<missing>').astype(str).value_counts().head(15)
        plt.figure(figsize=(9, 4))
        counts.sort_values().plot(kind='barh')
        plt.title(f'Top categories: {col}')
        plt.xlabel('Rows')
        plt.tight_layout()
        plt.show()

    if len(numeric_cols) >= 2:
        corr = df[numeric_cols].corr(numeric_only=True)
        plt.figure(figsize=(8, 6))
        image = plt.imshow(corr, vmin=-1, vmax=1, cmap='coolwarm')
        plt.colorbar(image, label='Correlation')
        plt.xticks(range(len(corr.columns)), corr.columns, rotation=60, ha='right')
        plt.yticks(range(len(corr.index)), corr.index)
        plt.title('Numeric correlation matrix')
        plt.tight_layout()
        plt.show()

    if len(numeric_cols) >= 2:
        x_col, y_col = numeric_cols[0], numeric_cols[-1]
        sample = df[[x_col, y_col]].dropna().sample(min(3000, len(df.dropna(subset=[x_col, y_col]))), random_state=42)
        if len(sample):
            plt.figure(figsize=(7, 5))
            plt.scatter(sample[x_col], sample[y_col], alpha=0.35, s=18)
            plt.xlabel(x_col)
            plt.ylabel(y_col)
            plt.title(f'{y_col} versus {x_col}')
            plt.tight_layout()
            plt.show()
else:
    print('Run the documented data-build/download path, then rerun this section to render raw-data EDA.')


## 4. Inspect the measured results, not just the code

A portfolio project is stronger when it retains evidence. This section reads machine-readable JSON/CSV outputs and turns scalar metrics into a quick visual comparison.


In [ ]:
json_files = [p for p in candidate_files if p.suffix.lower() == '.json']
metric_rows = []
for path in json_files[:30]:
    try:
        payload = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        continue
    stack = [('', payload)]
    while stack:
        prefix, value = stack.pop()
        if isinstance(value, dict):
            for key, child in value.items():
                stack.append((f'{prefix}.{key}' if prefix else str(key), child))
        elif isinstance(value, (int, float)) and not isinstance(value, bool) and np.isfinite(value):
            metric_rows.append({
                'file': str(path.relative_to(ROOT)) if ROOT in path.parents else str(path),
                'metric': prefix,
                'value': float(value),
            })
metrics_df = pd.DataFrame(metric_rows)
if len(metrics_df):
    display(metrics_df.head(40))
    plot_df = metrics_df[np.isfinite(metrics_df['value'])].copy()
    plot_df = plot_df[plot_df['value'].abs() < 1_000_000].head(20)
    if len(plot_df):
        labels = (plot_df['file'].str.split('/').str[-1] + ' :: ' + plot_df['metric']).tolist()
        plt.figure(figsize=(10, max(4, 0.35 * len(plot_df))))
        plt.barh(range(len(plot_df)), plot_df['value'])
        plt.yticks(range(len(plot_df)), labels)
        plt.title('Retained project metrics / evidence')
        plt.tight_layout()
        plt.show()
else:
    print('No scalar JSON evidence found. Run the project and retain metrics/results before treating it as complete.')


## 5. Reproduce the application

The notebook should be understandable without running anything, but a reviewer can reproduce the canonical application below. The switch is off by default so opening the notebook never triggers a long training job unexpectedly.


In [ ]:
RUN_PROJECT = False
entrypoint = PROJECT / 'run.py'
if RUN_PROJECT and entrypoint.exists():
    subprocess.run([sys.executable, str(entrypoint)], cwd=PROJECT, check=True)
elif entrypoint.exists():
    print(f'Reproduce with: cd {PROJECT} && {sys.executable} run.py')
else:
    print('This project uses a different documented entry point; see README.md in the project folder.')


## 6. Decision / solution

Route confident cases automatically and send ambiguous nearest-neighbour cases to review with the neighbour evidence visible.

The final recommendation should be tied to the measured validation evidence and error analysis below. A model is not the solution by itself; the solution is the decision process built around it.


# K-Nearest Neighbours — Product Quality Decision System

## Problem and objective
Build an interpretable distance-based classification application that predicts product-quality class, quantifies confidence, exposes nearest comparable observations, and sends uncertain predictions to manual review.


## Dataset and provenance
The project uses scikit-learn's built-in Wine dataset: 178 observations, 13 continuous chemical measurements and three classes. It is bundled through the sklearn dataset loader for deterministic reproduction. This is an educational quality-decision benchmark, not a claim about a live manufacturing process.


In [ ]:
from run import load_dataset, audit_dataset, descriptive_profile
x, y, target_names = load_dataset()
audit_dataset(x, y), x.head(), y.value_counts().sort_index()


## Analysis and validation
The full application uses leakage-safe scaling, stratified splitting, cross-validated KNN tuning, a scaling ablation, confusion-matrix/error analysis, permutation importance and confidence-based review logic. The canonical implementation is mirrored into this notebook by the portfolio workflow so the complete code remains visible here.


In [ ]:
from run import build_pipeline, scaling_ablation
baseline_knn = build_pipeline(n_neighbors=7)
scaling_ablation(x, y)


In [ ]:
from run import main
# Run the complete reproducible training/evaluation pipeline.
# main()


## Decision use, results and limitations
The application produces probability confidence, a manual-review flag and nearest-neighbour evidence. Results are written to `results/metrics.json` and model artefacts to `artifacts/`. Limitations include the compact benchmark size, sensitivity of distance methods to scaling and dimensionality, and the need to choose review thresholds from real operational costs. A production next step would validate on time-separated factory batches and monitor feature-distance drift.

## Reproducibility
Run `python run.py` from this project directory after installing the project requirements. Tests are in `tests/test_knn.py`.


## Interview discussion
Be ready to explain why KNN requires feature scaling, how `k` changes the bias/variance trade-off, why stratification matters, how neighbour distance can support interpretability, what the confidence-review threshold means operationally, and when a tree-based model would be a better choice.


# Deeper exploratory analysis and retained evidence

These direct notebook cells extend the initial EDA with data-quality, scale, relationship, output and error diagnostics. They are intentionally visible here rather than hidden behind project helper functions.


In [ ]:
# Extended data-quality scorecard
if df is not None and len(df):
    quality_rows = []
    for col in df.columns:
        series = df[col]
        row = {
            'feature': col,
            'dtype': str(series.dtype),
            'rows': len(series),
            'missing': int(series.isna().sum()),
            'missing_pct': float(100 * series.isna().mean()),
            'unique': int(series.nunique(dropna=False)),
            'unique_pct': float(100 * series.nunique(dropna=False) / max(len(series), 1)),
        }
        if pd.api.types.is_numeric_dtype(series):
            values = pd.to_numeric(series, errors='coerce').dropna()
            if len(values):
                q1, q3 = values.quantile([0.25, 0.75])
                iqr = q3 - q1
                row.update({
                    'mean': float(values.mean()),
                    'median': float(values.median()),
                    'std': float(values.std()),
                    'p05': float(values.quantile(0.05)),
                    'p95': float(values.quantile(0.95)),
                    'skew': float(values.skew()),
                    'iqr_outliers': int(((values < q1 - 1.5*iqr) | (values > q3 + 1.5*iqr)).sum()),
                })
        quality_rows.append(row)
    deep_quality = pd.DataFrame(quality_rows)
    display(deep_quality.sort_values(['missing_pct','unique'], ascending=[False,False]).head(40))
    if 'iqr_outliers' in deep_quality:
        outlier_view = deep_quality.dropna(subset=['iqr_outliers']).sort_values('iqr_outliers', ascending=False).head(15)
        if len(outlier_view):
            plt.figure(figsize=(10,4))
            plt.bar(outlier_view['feature'], outlier_view['iqr_outliers'])
            plt.title('Potential IQR outliers by feature')
            plt.ylabel('Rows')
            plt.xticks(rotation=60, ha='right')
            plt.tight_layout()
            plt.show()
    card = deep_quality.sort_values('unique', ascending=False).head(20)
    plt.figure(figsize=(10,4))
    plt.bar(card['feature'], card['unique'])
    plt.title('Feature cardinality')
    plt.ylabel('Unique values')
    plt.xticks(rotation=60, ha='right')
    plt.tight_layout()
    plt.show()
    print('Constant columns:', deep_quality.loc[deep_quality['unique'] <= 1, 'feature'].tolist())
    print('High-missing columns:', deep_quality.loc[deep_quality['missing_pct'] >= 30, 'feature'].tolist())
    print('Possible identifier columns:', deep_quality.loc[deep_quality['unique_pct'] >= 95, 'feature'].tolist()[:20])
else:
    print('Materialise the documented dataset to run the extended data-quality scorecard.')


In [ ]:
# Numeric distributions, spread and strongest pairwise relationships
if df is not None and len(df):
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()[:12]
    for col in numeric_cols:
        values = pd.to_numeric(df[col], errors='coerce').dropna()
        if len(values) < 5:
            continue
        clipped = values.clip(values.quantile(0.01), values.quantile(0.99))
        plt.figure(figsize=(8,4))
        plt.hist(clipped, bins=35, alpha=0.82)
        plt.axvline(values.median(), linestyle='--', label=f'median={values.median():.3g}')
        plt.axvline(values.mean(), linestyle=':', label=f'mean={values.mean():.3g}')
        plt.title(f'Distribution: {col} (1st–99th percentile)')
        plt.xlabel(col)
        plt.ylabel('Rows')
        plt.legend()
        plt.tight_layout()
        plt.show()
        plt.figure(figsize=(8,3))
        plt.boxplot(values, vert=False, showfliers=True)
        plt.title(f'Spread / outliers: {col}')
        plt.xlabel(col)
        plt.tight_layout()
        plt.show()
    if len(numeric_cols) >= 2:
        corr = df[numeric_cols].corr(numeric_only=True)
        pairs = []
        for i, left in enumerate(corr.columns):
            for right in corr.columns[i+1:]:
                value = corr.loc[left, right]
                if pd.notna(value):
                    pairs.append({'feature_a': left, 'feature_b': right, 'correlation': float(value), 'abs_correlation': float(abs(value))})
        corr_pairs = pd.DataFrame(pairs).sort_values('abs_correlation', ascending=False) if pairs else pd.DataFrame()
        if len(corr_pairs):
            display(corr_pairs.head(20).round(4))
            for _, pair in corr_pairs.head(4).iterrows():
                sample = df[[pair['feature_a'], pair['feature_b']]].dropna()
                if len(sample) > 3000:
                    sample = sample.sample(3000, random_state=42)
                plt.figure(figsize=(7,5))
                plt.scatter(sample[pair['feature_a']], sample[pair['feature_b']], alpha=0.30, s=16)
                plt.xlabel(pair['feature_a'])
                plt.ylabel(pair['feature_b'])
                plt.title(f"{pair['feature_a']} vs {pair['feature_b']} (r={pair['correlation']:.2f})")
                plt.tight_layout()
                plt.show()
    categorical = [c for c in df.columns if 2 <= df[c].nunique(dropna=False) <= 20][:8]
    for col in categorical:
        counts = df[col].fillna('<missing>').astype(str).value_counts().head(20)
        shares = 100 * counts / counts.sum()
        display(pd.DataFrame({'rows': counts, 'share_pct': shares.round(2)}))
        plt.figure(figsize=(8,4))
        counts.sort_values().plot(kind='barh')
        plt.title(f'Category balance: {col}')
        plt.xlabel('Rows')
        plt.tight_layout()
        plt.show()
else:
    print('Materialise the documented dataset to run distribution diagnostics.')


In [ ]:
# Temporal coverage where date/time fields exist
if df is not None and len(df):
    time_cols = [c for c in df.columns if any(token in str(c).lower() for token in ('date','time','timestamp','datetime'))]
    print('Date/time candidates:', time_cols[:10])
    for col in time_cols[:4]:
        converted = pd.to_datetime(df[col], errors='coerce')
        valid = converted.dropna()
        if len(valid) >= max(10, int(0.25*len(df))):
            print(col, 'range:', valid.min(), '→', valid.max())
            monthly = valid.dt.to_period('M').value_counts().sort_index()
            if len(monthly) > 1:
                plt.figure(figsize=(10,4))
                plt.plot(monthly.index.astype(str), monthly.values, marker='o')
                plt.title(f'Rows over time: {col}')
                plt.ylabel('Rows')
                plt.xticks(rotation=70, ha='right')
                plt.tight_layout()
                plt.show()


## Retained outputs and error analysis

A strong portfolio keeps inspectable evidence. The cells below profile compact result tables and automatically detect prediction-like columns for residual or misclassification analysis.


In [ ]:
# Load compact result/evidence tables
result_tables = []
for base in [PROJECT/'results', PROJECT/'outputs', PROJECT/'artifacts', ROOT/'verified'/PROJECT_SLUG]:
    if not base.exists():
        continue
    for path in sorted(base.rglob('*')):
        if path.is_file() and path.suffix.lower() in {'.csv','.tsv','.parquet'} and path.stat().st_size < 25_000_000:
            try:
                if path.suffix.lower() == '.parquet':
                    table = pd.read_parquet(path)
                else:
                    table = pd.read_csv(path, sep='	' if path.suffix.lower() == '.tsv' else ',')
            except Exception as exc:
                print('Could not read', path.name, '-', exc)
                continue
            result_tables.append((path, table))
            print('
RESULT TABLE:', path.relative_to(ROOT) if ROOT in path.parents else path)
            print('shape=', table.shape)
            display(table.head(15))
            numeric = table.select_dtypes(include=np.number).columns.tolist()[:12]
            if numeric:
                display(table[numeric].describe().T.round(4))
print('Inspectable result tables:', len(result_tables))


In [ ]:
# Automatic regression/classification-style error diagnostics
actual_tokens = ('actual','target','truth','y_true','observed','label')
pred_tokens = ('prediction','predicted','forecast','y_pred')
confidence_tokens = ('confidence','probability','proba','risk','uncertainty')
for path, table in result_tables:
    actual_cols = [c for c in table.columns if any(token in str(c).lower() for token in actual_tokens)]
    pred_cols = [c for c in table.columns if any(token in str(c).lower() for token in pred_tokens)]
    conf_cols = [c for c in table.columns if any(token in str(c).lower() for token in confidence_tokens)]
    if actual_cols and pred_cols and len(table):
        actual_col = actual_cols[0]
        pred_col = next((c for c in pred_cols if c != actual_col), pred_cols[0])
        actual_num = pd.to_numeric(table[actual_col], errors='coerce')
        pred_num = pd.to_numeric(table[pred_col], errors='coerce')
        numeric_mask = actual_num.notna() & pred_num.notna()
        if numeric_mask.sum() >= 10:
            residual = actual_num[numeric_mask] - pred_num[numeric_mask]
            abs_error = residual.abs()
            print('
', path.name, '| MAE=', round(float(abs_error.mean()),5), '| RMSE=', round(float(np.sqrt(np.mean(residual**2))),5), '| bias=', round(float(residual.mean()),5))
            plt.figure(figsize=(7,5))
            plt.scatter(actual_num[numeric_mask], pred_num[numeric_mask], alpha=0.35, s=18)
            lo = min(actual_num[numeric_mask].min(), pred_num[numeric_mask].min())
            hi = max(actual_num[numeric_mask].max(), pred_num[numeric_mask].max())
            plt.plot([lo,hi],[lo,hi], linestyle='--')
            plt.xlabel(str(actual_col))
            plt.ylabel(str(pred_col))
            plt.title(f'Actual vs predicted — {path.name}')
            plt.tight_layout()
            plt.show()
            plt.figure(figsize=(7,4))
            plt.hist(residual, bins=30, alpha=0.82)
            plt.axvline(0, linestyle='--')
            plt.title(f'Residual distribution — {path.name}')
            plt.tight_layout()
            plt.show()
            worst_idx = abs_error.nlargest(min(15,len(abs_error))).index
            cols = list(dict.fromkeys([actual_col,pred_col]+conf_cols[:2]))
            worst = table.loc[worst_idx, cols].copy()
            worst['absolute_error'] = abs_error.loc[worst_idx].values
            display(worst.sort_values('absolute_error', ascending=False))
        else:
            agreement = table[actual_col].astype(str) == table[pred_col].astype(str)
            print('
', path.name, '| classification agreement=', round(float(agreement.mean()),4))
            if (~agreement).any():
                display(table.loc[~agreement, [actual_col,pred_col]+conf_cols[:2]].head(20))
    elif conf_cols:
        for col in conf_cols[:2]:
            values = pd.to_numeric(table[col], errors='coerce').dropna()
            if len(values) >= 10:
                plt.figure(figsize=(7,4))
                plt.hist(values, bins=30, alpha=0.82)
                plt.title(f'{col} distribution — {path.name}')
                plt.tight_layout()
                plt.show()


In [ ]:
# Display retained visual evidence from actual project runs
png_files = []
for base in [PROJECT/'results', PROJECT/'outputs', PROJECT/'artifacts', ROOT/'verified'/PROJECT_SLUG]:
    if base.exists():
        png_files.extend(sorted(base.rglob('*.png')))
print('Retained PNG figures:', len(png_files))
for path in png_files[:12]:
    try:
        image = plt.imread(path)
        plt.figure(figsize=(10,6))
        plt.imshow(image)
        plt.axis('off')
        plt.title(str(path.relative_to(ROOT)) if ROOT in path.parents else path.name)
        plt.tight_layout()
        plt.show()
    except Exception as exc:
        print('Could not display', path.name, '-', exc)


In [ ]:
# Reproducibility and evidence checklist
checks = [
    {'check':'README present', 'status':(PROJECT/'README.md').exists()},
    {'check':'Recruiter notebook present', 'status':(PROJECT/'project_notebook.ipynb').exists()},
    {'check':'Python implementation present', 'status':any(PROJECT.rglob('*.py'))},
    {'check':'Tests present', 'status':(PROJECT/'tests').exists() and any((PROJECT/'tests').rglob('test*.py'))},
    {'check':'Result/evidence files present', 'status':bool(candidate_files)},
    {'check':'Machine-readable JSON evidence', 'status':bool(json_files)},
    {'check':'Retained visual evidence', 'status':bool(png_files)},
]
checklist = pd.DataFrame(checks)
display(checklist)
print('Evidence checklist pass rate:', f"{100*checklist['status'].mean():.1f}%")
print('A failed item is a prompt to strengthen the project, not something to hide.')


# Robustness, slices and decision analysis

A model or pipeline is useful only when we know where it works, where it fails and what action follows. This section adds direct slice analysis, sensitivity checks and a compact decision memo from the evidence already produced by the project.


In [ ]:
# Quantile slices for important numeric variables
if df is not None and len(df):
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()[:10]
    quantile_rows = []
    for col in numeric_cols:
        values = pd.to_numeric(df[col], errors='coerce')
        valid = values.dropna()
        if len(valid) < 20 or valid.nunique() < 5:
            continue
        quantiles = valid.quantile([0.01,0.05,0.10,0.25,0.50,0.75,0.90,0.95,0.99])
        for q, value in quantiles.items():
            quantile_rows.append({'feature':col, 'quantile':q, 'value':float(value)})
    quantile_table = pd.DataFrame(quantile_rows)
    if len(quantile_table):
        display(quantile_table.pivot(index='feature', columns='quantile', values='value').round(4))
        for col in quantile_table['feature'].unique()[:6]:
            view = quantile_table[quantile_table['feature']==col]
            plt.figure(figsize=(7,4))
            plt.plot(view['quantile'], view['value'], marker='o')
            plt.xlabel('Quantile')
            plt.ylabel(col)
            plt.title(f'Quantile profile: {col}')
            plt.tight_layout()
            plt.show()
else:
    print('Quantile slices become available after the project dataset is materialised.')


In [ ]:
# Missingness and duplication sensitivity
if df is not None and len(df):
    missing_by_row = df.isna().sum(axis=1)
    print('Rows with any missing value:', int((missing_by_row>0).sum()))
    print('Rows with 2+ missing values:', int((missing_by_row>=2).sum()))
    print('Exact duplicate rows:', int(df.duplicated().sum()))
    if missing_by_row.max() > 0:
        plt.figure(figsize=(7,4))
        missing_by_row.value_counts().sort_index().plot(kind='bar')
        plt.title('Missing cells per row')
        plt.xlabel('Missing cells')
        plt.ylabel('Rows')
        plt.tight_layout()
        plt.show()
    duplicated = df.duplicated(keep=False)
    if duplicated.any():
        display(df.loc[duplicated].head(20))
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()[:10]
    robust_rows = []
    for col in numeric_cols:
        values = pd.to_numeric(df[col], errors='coerce').dropna()
        if len(values) < 20:
            continue
        median = values.median()
        mad = np.median(np.abs(values-median))
        robust_z = 0.6745*(values-median)/(mad if mad else 1.0)
        robust_rows.append({'feature':col, 'median':median, 'mad':mad, 'robust_outliers_abs_z_gt_3_5':int((np.abs(robust_z)>3.5).sum())})
    robust_outliers = pd.DataFrame(robust_rows).sort_values('robust_outliers_abs_z_gt_3_5', ascending=False) if robust_rows else pd.DataFrame()
    if len(robust_outliers):
        display(robust_outliers.round(4))


In [ ]:
# Concentration / imbalance analysis for important categorical dimensions
if df is not None and len(df):
    categorical = [c for c in df.columns if 2 <= df[c].nunique(dropna=False) <= 50][:10]
    concentration_rows = []
    for col in categorical:
        counts = df[col].fillna('<missing>').astype(str).value_counts()
        shares = counts / counts.sum()
        hhi = float((shares**2).sum())
        concentration_rows.append({'feature':col, 'categories':len(counts), 'largest_share':float(shares.iloc[0]), 'top3_share':float(shares.head(3).sum()), 'hhi':hhi})
    concentration = pd.DataFrame(concentration_rows).sort_values('hhi', ascending=False) if concentration_rows else pd.DataFrame()
    if len(concentration):
        display(concentration.round(4))
        plt.figure(figsize=(9,4))
        plt.bar(concentration['feature'], concentration['largest_share'])
        plt.ylabel('Largest category share')
        plt.title('Category concentration / imbalance')
        plt.xticks(rotation=60, ha='right')
        plt.tight_layout()
        plt.show()


In [ ]:
# Rank all retained scalar metrics and highlight likely success/risk signals
metric_records = []
for path in json_files[:60]:
    try:
        payload = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        continue
    stack = [('', payload)]
    while stack:
        prefix, value = stack.pop()
        if isinstance(value, dict):
            for key, child in value.items():
                stack.append((f'{prefix}.{key}' if prefix else str(key), child))
        elif isinstance(value, (int,float)) and not isinstance(value,bool) and np.isfinite(value):
            metric_records.append({'file':path.name, 'metric':prefix, 'value':float(value)})
all_metrics = pd.DataFrame(metric_records)
if len(all_metrics):
    signal_pattern = 'accuracy|f1|auc|precision|recall|r2|rmse|mae|loss|coverage|review|drift|psi|brier|calibration|revenue|cost|effect|lift|latency|row|reject|duplicate'
    decision_metrics = all_metrics[all_metrics['metric'].str.contains(signal_pattern, case=False, regex=True)].copy()
    if not len(decision_metrics):
        decision_metrics = all_metrics.copy()
    decision_metrics = decision_metrics.drop_duplicates(['file','metric']).reset_index(drop=True)
    display(decision_metrics.head(60).round(6))
    rate_like = decision_metrics[decision_metrics['metric'].str.contains('accuracy|f1|auc|precision|recall|coverage|rate|r2', case=False, regex=True)]
    if len(rate_like):
        bounded = rate_like[(rate_like['value']>=-1)&(rate_like['value']<=1)].head(30)
        if len(bounded):
            plt.figure(figsize=(10,max(5,0.3*len(bounded))))
            plt.barh(range(len(bounded)), bounded['value'])
            plt.yticks(range(len(bounded)), bounded['file']+' :: '+bounded['metric'])
            plt.xlim(min(-0.05,bounded['value'].min()-0.05),1.05)
            plt.title('Retained rate / quality metrics')
            plt.tight_layout()
            plt.show()
    error_like = decision_metrics[decision_metrics['metric'].str.contains('rmse|mae|loss|error|latency|drift|psi|brier', case=False, regex=True)]
    if len(error_like):
        display(error_like.sort_values('value', ascending=False).head(30).round(6))
else:
    print('No retained scalar JSON metrics are available yet.')


In [ ]:
# Inspect artifact sizes — a quick engineering sanity check
artifact_rows = []
for base in [PROJECT/'artifacts', PROJECT/'results', PROJECT/'outputs', ROOT/'verified'/PROJECT_SLUG]:
    if not base.exists():
        continue
    for path in base.rglob('*'):
        if path.is_file():
            artifact_rows.append({'file':str(path.relative_to(ROOT)) if ROOT in path.parents else str(path), 'suffix':path.suffix.lower(), 'size_kb':path.stat().st_size/1024})
artifacts_df = pd.DataFrame(artifact_rows).sort_values('size_kb', ascending=False) if artifact_rows else pd.DataFrame()
if len(artifacts_df):
    display(artifacts_df.head(40).round(2))
    by_type = artifacts_df.groupby('suffix', as_index=False).agg(files=('file','size'), total_kb=('size_kb','sum')).sort_values('total_kb', ascending=False)
    display(by_type.round(2))
    plt.figure(figsize=(8,4))
    plt.bar(by_type['suffix'].replace('', '<none>'), by_type['total_kb'])
    plt.ylabel('Total KB')
    plt.title('Retained evidence by file type')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print('No retained artifacts/results found.')


In [ ]:
# Threshold / coverage trade-off when a result table contains confidence or probability
for path, table in result_tables:
    conf_cols = [c for c in table.columns if any(token in str(c).lower() for token in ('confidence','probability','proba','score','risk'))]
    correct_cols = [c for c in table.columns if 'correct' in str(c).lower()]
    if not conf_cols or not len(table):
        continue
    confidence = pd.to_numeric(table[conf_cols[0]], errors='coerce')
    valid_conf = confidence.notna()
    if valid_conf.sum() < 20:
        continue
    trade_rows = []
    for threshold in np.linspace(float(confidence[valid_conf].quantile(0.10)), float(confidence[valid_conf].quantile(0.90)), 9):
        accepted = valid_conf & (confidence >= threshold)
        row = {'threshold':float(threshold), 'coverage':float(accepted.mean()), 'review_rate':float((valid_conf & ~accepted).sum()/valid_conf.sum()), 'accepted_rows':int(accepted.sum())}
        if correct_cols:
            correctness = table[correct_cols[0]].astype(bool)
            row['accepted_accuracy'] = float(correctness[accepted].mean()) if accepted.any() else np.nan
        trade_rows.append(row)
    trade = pd.DataFrame(trade_rows)
    print('Trade-off table from', path.name, 'using', conf_cols[0])
    display(trade.round(4))
    plt.figure(figsize=(8,4))
    plt.plot(trade['threshold'], trade['coverage'], marker='o', label='coverage')
    if 'accepted_accuracy' in trade:
        plt.plot(trade['threshold'], trade['accepted_accuracy'], marker='o', label='accepted accuracy')
    plt.xlabel('Threshold')
    plt.ylabel('Rate')
    plt.title(f'Threshold trade-off — {path.name}')
    plt.legend()
    plt.tight_layout()
    plt.show()
    break


In [ ]:
# Produce a concise evidence-backed decision memo inside the notebook
project_summary = {
    'project': PROJECT_SLUG,
    'local_data_or_evidence_files': int(len(candidate_files)),
    'result_tables': int(len(result_tables)),
    'json_evidence_files': int(len(json_files)),
    'visual_evidence_files': int(len(png_files)),
    'has_tests': bool((PROJECT/'tests').exists() and any((PROJECT/'tests').rglob('test*.py'))),
    'has_readme': bool((PROJECT/'README.md').exists()),
}
if df is not None:
    project_summary.update({'inspected_rows':int(len(df)), 'inspected_columns':int(df.shape[1]), 'duplicate_rows':int(df.duplicated().sum()), 'missing_cells':int(df.isna().sum().sum())})
summary_table = pd.DataFrame({'item':list(project_summary.keys()), 'value':list(project_summary.values())})
display(summary_table)
print('DECISION PRINCIPLE')
print('1. Use the measured evidence above, not model complexity, to choose the final approach.')
print('2. Inspect the worst slices/failures before making a business or operational recommendation.')
print('3. Keep uncertain, novel or high-impact cases on a review/escalation path where appropriate.')
print('4. Treat the documented limitations as part of the solution, not as boilerplate.')


# Engineering appendix — canonical application source

The analysis and visual evidence come first. The cells below preserve additional canonical Python from this project for reviewers who want to inspect pipelines, APIs, tests, feature code, monitoring and reusable implementation details.


## Canonical source: `run.py`


In [ ]:
from __future__ import annotations

import json
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import joblib
import numpy as np
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
ROOT = Path(__file__).resolve().parent
RESULTS = ROOT / "results"
ARTIFACTS = ROOT / "artifacts"
RESULTS.mkdir(exist_ok=True)
ARTIFACTS.mkdir(exist_ok=True)


@dataclass
class DatasetAudit:
    rows: int
    columns: int
    target_classes: int
    duplicate_rows: int
    missing_cells: int
    min_class_size: int
    max_class_size: int


@dataclass
class Evaluation:
    accuracy: float
    balanced_accuracy: float
    macro_f1: float
    log_loss: float
    review_rate: float
    accepted_accuracy: float | None


def load_dataset() -> tuple[pd.DataFrame, pd.Series, list[str]]:
    bunch = load_wine(as_frame=True)
    frame = bunch.frame.copy()
    target = frame.pop("target").astype(int)
    names = [str(name) for name in bunch.target_names]
    frame.columns = [str(c).strip().lower().replace(" ", "_") for c in frame.columns]
    return frame, target, names


def audit_dataset(x: pd.DataFrame, y: pd.Series) -> DatasetAudit:
    counts = y.value_counts()
    audit = DatasetAudit(
        rows=len(x),
        columns=x.shape[1],
        target_classes=int(y.nunique()),
        duplicate_rows=int(x.duplicated().sum()),
        missing_cells=int(x.isna().sum().sum()),
        min_class_size=int(counts.min()),
        max_class_size=int(counts.max()),
    )
    if audit.rows < 100:
        raise ValueError("Dataset unexpectedly small")
    if audit.target_classes < 2:
        raise ValueError("Classification requires multiple classes")
    if audit.missing_cells:
        raise ValueError("Built-in benchmark should not contain missing values")
    return audit


def descriptive_profile(x: pd.DataFrame, y: pd.Series) -> dict[str, Any]:
    profile: dict[str, Any] = {
        "shape": [int(x.shape[0]), int(x.shape[1])],
        "class_distribution": {str(k): int(v) for k, v in y.value_counts().sort_index().items()},
        "feature_means": {k: float(v) for k, v in x.mean().items()},
        "feature_std": {k: float(v) for k, v in x.std().items()},
        "feature_min": {k: float(v) for k, v in x.min().items()},
        "feature_max": {k: float(v) for k, v in x.max().items()},
    }
    return profile


def build_pipeline(n_neighbors: int = 5, weights: str = "distance", p: int = 2) -> Pipeline:
    return Pipeline(
        steps=[
            ("scale", StandardScaler()),
            ("model", KNeighborsClassifier(n_neighbors=n_neighbors, weights=weights, p=p)),
        ]
    )


def tune_model(x_train: pd.DataFrame, y_train: pd.Series) -> GridSearchCV:
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    search = GridSearchCV(
        estimator=build_pipeline(),
        param_grid={
            "model__n_neighbors": list(range(3, 22, 2)),
            "model__weights": ["uniform", "distance"],
            "model__p": [1, 2],
        },
        scoring="f1_macro",
        cv=cv,
        n_jobs=-1,
        return_train_score=True,
    )
    search.fit(x_train, y_train)
    return search


def evaluate_confidence_policy(
    probabilities: np.ndarray,
    predictions: np.ndarray,
    truth: np.ndarray,
    threshold: float,
) -> tuple[float, float | None]:
    confidence = probabilities.max(axis=1)
    accepted = confidence >= threshold
    review_rate = float((~accepted).mean())
    accepted_accuracy = None
    if accepted.any():
        accepted_accuracy = float(accuracy_score(truth[accepted], predictions[accepted]))
    return review_rate, accepted_accuracy


def evaluate_model(model: Pipeline, x_test: pd.DataFrame, y_test: pd.Series) -> tuple[Evaluation, dict[str, Any]]:
    predictions = model.predict(x_test)
    probabilities = model.predict_proba(x_test)
    review_rate, accepted_accuracy = evaluate_confidence_policy(
        probabilities,
        predictions,
        y_test.to_numpy(),
        threshold=0.70,
    )
    evaluation = Evaluation(
        accuracy=float(accuracy_score(y_test, predictions)),
        balanced_accuracy=float(balanced_accuracy_score(y_test, predictions)),
        macro_f1=float(f1_score(y_test, predictions, average="macro")),
        log_loss=float(log_loss(y_test, probabilities)),
        review_rate=review_rate,
        accepted_accuracy=accepted_accuracy,
    )
    detail = {
        "confusion_matrix": confusion_matrix(y_test, predictions).tolist(),
        "classification_report": classification_report(y_test, predictions, output_dict=True),
        "confidence": probabilities.max(axis=1).tolist(),
    }
    return evaluation, detail


def scaling_ablation(x_train: pd.DataFrame, y_train: pd.Series) -> dict[str, float]:
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    scaled = build_pipeline(n_neighbors=7)
    unscaled = KNeighborsClassifier(n_neighbors=7, weights="distance")
    scaled_score = cross_val_score(scaled, x_train, y_train, scoring="f1_macro", cv=cv).mean()
    raw_score = cross_val_score(unscaled, x_train, y_train, scoring="f1_macro", cv=cv).mean()
    return {
        "scaled_macro_f1": float(scaled_score),
        "unscaled_macro_f1": float(raw_score),
        "scaling_gain": float(scaled_score - raw_score),
    }


def feature_importance(model: Pipeline, x_test: pd.DataFrame, y_test: pd.Series) -> list[dict[str, float | str]]:
    result = permutation_importance(
        model,
        x_test,
        y_test,
        scoring="f1_macro",
        n_repeats=25,
        random_state=RANDOM_STATE,
    )
    rows = [
        {"feature": feature, "importance": float(score)}
        for feature, score in zip(x_test.columns, result.importances_mean)
    ]
    return sorted(rows, key=lambda row: float(row["importance"]), reverse=True)


def inspect_neighbours(model: Pipeline, x: pd.DataFrame, row: pd.DataFrame, top_n: int = 5) -> list[dict[str, Any]]:
    scaler: StandardScaler = model.named_steps["scale"]
    knn: KNeighborsClassifier = model.named_steps["model"]
    x_scaled = scaler.transform(x)
    row_scaled = scaler.transform(row)
    distances, indices = knn.kneighbors(row_scaled, n_neighbors=top_n)
    output: list[dict[str, Any]] = []
    for distance, idx in zip(distances[0], indices[0]):
        output.append({"row_index": int(idx), "distance": float(distance)})
    return output


def predict_one(model: Pipeline, row: pd.DataFrame, target_names: list[str]) -> dict[str, Any]:
    probabilities = model.predict_proba(row)[0]
    prediction = int(model.predict(row)[0])
    confidence = float(probabilities.max())
    return {
        "predicted_class": prediction,
        "predicted_label": target_names[prediction],
        "confidence": confidence,
        "manual_review": bool(confidence < 0.70),
        "class_probabilities": {
            target_names[i]: float(probabilities[i]) for i in range(len(probabilities))
        },
    }


def save_json(path: Path, payload: Any) -> None:
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")


def main() -> None:
    x, y, target_names = load_dataset()
    audit = audit_dataset(x, y)
    profile = descriptive_profile(x, y)

    x_train, x_test, y_train, y_test = train_test_split(
        x,
        y,
        test_size=0.25,
        stratify=y,
        random_state=RANDOM_STATE,
    )

    search = tune_model(x_train, y_train)
    best_model: Pipeline = search.best_estimator_
    evaluation, evaluation_detail = evaluate_model(best_model, x_test, y_test)
    ablation = scaling_ablation(x_train, y_train)
    importance = feature_importance(best_model, x_test, y_test)
    example = predict_one(best_model, x_test.iloc[[0]], target_names)
    neighbours = inspect_neighbours(best_model, x_train, x_test.iloc[[0]], top_n=5)

    cv_table = pd.DataFrame(search.cv_results_).sort_values("rank_test_score")
    cv_table[
        ["params", "mean_test_score", "std_test_score", "mean_train_score", "rank_test_score"]
    ].head(20).to_csv(RESULTS / "cv_results.csv", index=False)

    payload = {
        "dataset_audit": asdict(audit),
        "evaluation": asdict(evaluation),
        "best_parameters": search.best_params_,
        "best_cv_macro_f1": float(search.best_score_),
        "scaling_ablation": ablation,
        "top_permutation_features": importance[:10],
        "example_prediction": example,
        "example_neighbours": neighbours,
        "confusion_matrix": evaluation_detail["confusion_matrix"],
        "limitations": [
            "Compact benchmark dataset rather than a live production quality stream.",
            "Neighbour distances can become less informative as dimensionality grows.",
            "A production threshold should be chosen from explicit quality-review costs.",
        ],
    }
    save_json(RESULTS / "metrics.json", payload)
    save_json(RESULTS / "dataset_profile.json", profile)
    joblib.dump(best_model, ARTIFACTS / "knn_quality_pipeline.joblib")

    reloaded: Pipeline = joblib.load(ARTIFACTS / "knn_quality_pipeline.joblib")
    original_pred = best_model.predict(x_test)
    reloaded_pred = reloaded.predict(x_test)
    if not np.array_equal(original_pred, reloaded_pred):
        raise RuntimeError("Saved-model parity check failed")

    print(json.dumps(payload, indent=2))


if __name__ == "__main__":
    main()


## Canonical source: `tests/test_knn.py`


In [ ]:
from pathlib import Path
import sys

import numpy as np

PROJECT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(PROJECT))

from run import audit_dataset, build_pipeline, load_dataset, predict_one


def test_dataset_contract():
    x, y, names = load_dataset()
    audit = audit_dataset(x, y)
    assert audit.rows == len(x)
    assert audit.columns == x.shape[1]
    assert audit.target_classes == 3
    assert len(names) == 3


def test_pipeline_returns_probabilities():
    x, y, names = load_dataset()
    model = build_pipeline(n_neighbors=5)
    model.fit(x, y)
    result = predict_one(model, x.iloc[[0]], names)
    probabilities = np.array(list(result["class_probabilities"].values()))
    assert np.isclose(probabilities.sum(), 1.0)
    assert 0.0 <= result["confidence"] <= 1.0


# Portfolio depth check

**Meaningful visible code lines after all notebook passes:** 839. The working target for a major application is roughly 1,000 meaningful lines when justified by the problem. This notebook is in/above the working depth range. Line count is never permission to add filler; depth must come from data, analysis, visualisation, modelling/engineering, evaluation, robustness and decision logic.
